In [1]:
!pip install -q \
transformers \
accelerate \
openpyxl \
scikit-learn

In [2]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [3]:
from google.colab import files

uploaded = files.upload()

Saving sinhala_youtube_comments_for_annotation.xlsx to sinhala_youtube_comments_for_annotation.xlsx


In [4]:
df = pd.read_excel(
    "/content/sinhala_youtube_comments_for_annotation.xlsx"
)

In [5]:
print(df.columns)

Index(['id', 'comment', 'label'], dtype='object')


In [6]:
df = df[['comment', 'label']]

In [7]:
df.dropna(inplace=True)

In [8]:
df['label'] = df['label'].astype(int)

In [9]:
print(df.head())

print(df['label'].value_counts())

                                             comment  label
0                                           පියතුමනි      0
1                                       නියම යි❤❤❤❤❤      0
2                              හිච්චගේ මුණ තමා maru😂      0
3  සානක අයියගෙ වීඩියෝ බලනවා නම් හිනා වෙන්න පුලුවන...      1
4  අයියා වෙඩින් මුද්ද නම් දාගෙනම නේද ටික් ටොක් කර...      0
label
0    1577
1    1471
Name: count, dtype: int64


In [10]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['comment'].astype(str).tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

In [11]:
tokenizer = AutoTokenizer.from_pretrained(
    "keshan/SinhalaBERTo"
)

model = AutoModelForSequenceClassification.from_pretrained(
    "keshan/SinhalaBERTo",
    num_labels=2
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/721k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/334M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: keshan/SinhalaBERTo
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=128
)

In [13]:
class SarcasmDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):

        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item['labels'] = torch.tensor(
            self.labels[idx]
        )

        return item

    def __len__(self):

        return len(self.labels)

In [14]:
train_dataset = SarcasmDataset(
    train_encodings,
    train_labels
)

test_dataset = SarcasmDataset(
    test_encodings,
    test_labels
)

In [15]:
def compute_metrics(pred):

    labels = pred.label_ids

    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average='macro'
    )

    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [17]:
training_args = TrainingArguments(

    output_dir="./results",

    num_train_epochs=5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    warmup_steps=100,

    weight_decay=0.01,

    logging_dir="./logs",

    logging_steps=10,

    save_total_limit=1
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [18]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

In [19]:
trainer.train()

Step,Training Loss
10,0.727947
20,0.692944
30,0.681647
40,0.691216
50,0.670082
60,0.715597
70,0.704657
80,0.699450
90,0.734274
100,0.624599


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1525, training_loss=0.34686191172018405, metrics={'train_runtime': 235.825, 'train_samples_per_second': 51.691, 'train_steps_per_second': 6.467, 'total_flos': 403694397404160.0, 'train_loss': 0.34686191172018405, 'epoch': 5.0})

In [20]:
results = trainer.evaluate()

print(results)

Training Loss,Validation Loss,Step,Accuracy,F1,Precision,Recall
0.072420,1.934768,1525,0.681967,0.681296,0.681526,0.681198


{'eval_loss': 1.9347676038742065, 'eval_accuracy': 0.6819672131147541, 'eval_f1': 0.6812957157784744, 'eval_precision': 0.6815260524499656, 'eval_recall': 0.6811977955739257}


In [21]:
predictions = trainer.predict(test_dataset)

preds = predictions.predictions.argmax(-1)

cm = confusion_matrix(test_labels, preds)

print(cm)

[[222  94]
 [100 194]]


In [22]:
print(
    classification_report(
        test_labels,
        preds
    )
)

              precision    recall  f1-score   support

           0       0.69      0.70      0.70       316
           1       0.67      0.66      0.67       294

    accuracy                           0.68       610
   macro avg       0.68      0.68      0.68       610
weighted avg       0.68      0.68      0.68       610



In [23]:
model.save_pretrained(
    "./sarcasm_model"
)

tokenizer.save_pretrained(
    "./sarcasm_model"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./sarcasm_model/tokenizer_config.json', './sarcasm_model/tokenizer.json')

In [24]:
!zip -r sarcasm_model.zip sarcasm_model

  adding: sarcasm_model/ (stored 0%)
  adding: sarcasm_model/tokenizer.json (deflated 83%)
  adding: sarcasm_model/tokenizer_config.json (deflated 50%)
  adding: sarcasm_model/config.json (deflated 50%)
  adding: sarcasm_model/model.safetensors (deflated 7%)


In [1]:
from google.colab import files

files.download("sarcasm_model.zip")

FileNotFoundError: Cannot find file: sarcasm_model.zip

In [26]:
text = "අද නම් හරිම ලස්සන වැඩක් කරලා 😂"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

outputs = model(**inputs)

prediction = torch.argmax(
    outputs.logits
)

print(prediction.item())

RuntimeError: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)

In [27]:
text = "අද නම් හරිම ලස්සන වැඩක් කරලා 😂"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

inputs = {key: value.to(device) for key, value in inputs.items()}

outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1)

print("Prediction:", prediction.item())

if prediction.item() == 1:
    print("Sarcastic")
else:
    print("Non-Sarcastic")

Prediction: 0
Non-Sarcastic
